# 39 — Embeddings & Similarity

### Why this exists
Similarity is a retrieval signal, not proof of relevance. This lab compares lexical overlap with a deterministic vector-like representation and exposes a high-similarity-but-wrong result.

### Learning objectives
represent text vectors; calculate cosine similarity; compare ranking signals; set thresholds

### Prerequisite
Python math and basic vector reasoning

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/39-embeddings-similarity.svg)

> **Question to carry through the lab:** Can the most similar document still be the wrong document? What evidence would reveal that?

## Capability contract

**Capability:** C08 — LLM Systems  
**Workstream:** AI Support Platform  
**Primary roles:** AI Engineer|ML Engineer  
**You will prove:** Build or inspect a grounded AI mechanism


## Simulation → reality bridge

**Real-system bridge:** Embedding and vector retrieval systems

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook computes deterministic similarity to isolate representation and ranking. Real embedding services/models add dimensionality, model/version changes, normalization choices, latency and index behavior.

**Transfer challenge:** Compare a retrieval failure caused by representation with one caused by index/ranking configuration and state the evidence that separates them.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


## Mental model
Embeddings compress semantic information into a geometry. Retrieval still needs corpus quality, ranking strategy, thresholds and evaluation; similarity scores are not truth scores.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S18 — 1201: Policy search misses paraphrased customer questions** → [`../work_simulations/18_acme_commerce_embedding_search/README.md`](../work_simulations/18_acme_commerce_embedding_search/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
import math
def vec(text):
    words=text.lower().split(); vocab=sorted(set(words)); return [words.count(w) for w in vocab]
def cosine(a,b):
    vocab=sorted(set(a.lower().split())|set(b.lower().split()))
    av=[a.lower().split().count(w) for w in vocab]; bv=[b.lower().split().count(w) for w in vocab]
    d=sum(x*y for x,y in zip(av,bv)); na=math.sqrt(sum(x*x for x in av)); nb=math.sqrt(sum(y*y for y in bv)); return d/(na*nb) if na and nb else 0
query="refund payment"
docs=["refund payment policy", "payment refund fraud", "payment gateway latency", "shipping refund address"]
print([(d,round(cosine(query,d),3)) for d in docs])

In [ ]:
# Rank by the signal, then inspect relevance separately.
ranked=sorted(((d,cosine(query,d)) for d in docs),key=lambda x:x[1],reverse=True)
print(ranked)

In [ ]:
relevant={"refund payment policy","payment refund fraud"}
for d,s in ranked: print(d,"retrieved",d in relevant,"score",round(s,3))

In [ ]:
# Break: lower the threshold and observe recall/precision tension.
for threshold in [0.1,0.3,0.6]:
    selected=[d for d,s in ranked if s>=threshold]
    print(threshold,selected)

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## Your task

**Goal:** Measure retrieval quality, not just similarity.

**Do this:** Create a false-positive document, compare top-k and threshold retrieval, and explain why labeled relevance is needed.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 8. Production bridge
Production retrieval combines indexing, ranking, filtering, thresholds, freshness and evaluation; similarity alone cannot establish groundedness.

## 9. Independent challenge
Create a false positive with an irrelevant document sharing many words. Compare top-k and threshold retrieval and explain why evaluation must include labeled relevance.

### Evidence to keep
ranking scores, relevance labels, threshold behavior and a retrieval-quality decision

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

---

**Check your work:** [Open the reference solution](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/39.md) — only after you have attempted the exercise.